# Home Cost Pulse: test your API keys

Run each cell (Shift+Enter). Your keys are typed into a hidden box and are not saved in this notebook.

1. EIA key: https://www.eia.gov/opendata/register.php
2. NLR PVWatts key: https://developer.nlr.gov/signup/

In [ ]:
from getpass import getpass
EIA_API_KEY = getpass('EIA API key: ').strip()
NLR_API_KEY = getpass('NLR (PVWatts) API key: ').strip()

In [ ]:
import requests

params = [
    ('api_key', EIA_API_KEY), ('frequency', 'monthly'),
    ('data[0]', 'price'), ('data[1]', 'revenue'), ('data[2]', 'sales'), ('data[3]', 'customers'),
    ('facets[sectorid][]', 'RES'),
    ('facets[stateid][]', 'TX'), ('facets[stateid][]', 'CA'), ('facets[stateid][]', 'US'),
    ('sort[0][column]', 'period'), ('sort[0][direction]', 'desc'), ('length', '6'),
]
r = requests.get('https://api.eia.gov/v2/electricity/retail-sales/data/', params=params, timeout=30)
print('HTTP', r.status_code)
rows = r.json().get('response', {}).get('data', [])
for row in rows:
    usage = float(row['sales']) * 1e6 / float(row['customers']) if row.get('customers') else None
    bill = float(row['revenue']) * 1e6 / float(row['customers']) if row.get('customers') else None
    print(f"{row['period']}  {row['stateid']:>2}  {row['price']:>6} cents/kWh   "
          f"usage {usage:,.0f} kWh   bill ${bill:,.2f}" if usage else row)
print('EIA key works' if rows else 'No rows: check the key')

In [ ]:
p = {'api_key': NLR_API_KEY, 'system_capacity': 1, 'module_type': 0, 'losses': 14, 'array_type': 1,
     'tilt': 20, 'azimuth': 180, 'lat': 29.76, 'lon': -95.37, 'dataset': 'nsrdb', 'radius': 0}
for host in ('https://developer.nlr.gov',):  # developer.nrel.gov stopped working on May 29, 2026
    r = requests.get(host + '/api/pvwatts/v8.json', params=p, timeout=60)
    print(host, 'HTTP', r.status_code)
    if r.ok:
        out = r.json()['outputs']
        print(f"Houston: {out['ac_annual']:,.0f} kWh per kW per year, capacity factor {out['capacity_factor']:.1f}%")
        print('NLR key works')
        break

If both cells print that the key works, add them in GitHub: **Settings > Secrets and variables > Actions > New repository secret** with the names `EIA_API_KEY` and `NLR_API_KEY`.